# News To Stock Analyser 데이터준비

뉴스기사를 전달하면, 긍/부정분석 뿐아니라, 특정주식에 대한 긍/부정평가 처리 RAG 구현

In [1]:
%pip install -Uq datasets langchian-openai

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchian-openai (from versions: none)
ERROR: No matching distribution found for langchian-openai

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 데이터준비
https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko

In [2]:
from datasets import load_dataset

dataset = load_dataset('daekeun-ml/naver-news-summarization-ko')
dataset

c:\Users\Playdata\llm\llm_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [3]:
economy_dataset = dataset['train'].filter(lambda row: row['category'] == 'economy')
print(len(economy_dataset))

17088


In [4]:
economy_dataset[10]

{'date': '2022-07-01 06:54:01',
 'category': 'economy',
 'press': 'SBS Biz ',
 'title': '글로벌 비즈 가트너 올해 전세계 스마트폰 판매량 7% 감소 전망',
 'document': '경제와이드 모닝벨 글로벌 비즈 임선우 외신캐스터 글로벌 비즈입니다. ◇ 올해 스마트폰 판매 감소 올해 전세계 스마트폰 판매량이 크게 줄어들 것이란 전망이 나왔습니다. 시장조사업체 가트너는 글로벌 스마트폰 판매가 7% 하락할 것으로 내다봤는데요. 경제 전반에 걸친 침체 우려와 중국의 봉쇄조치 여파 그리고 인플레이션으로 소비자들이 지갑을 열기 주저하면서 수요가 줄어들 것 이라고 설명했습니다. 그러면서 올해 전체 출하량은 14억6천만대 수준에 그칠 것으로 예측했는데요. 종전 전망치인 16억대에서 대폭 낮춰 잡았습니다. 특히 세계 최대 스마트폰 시장인 중국에서 판매량은 18%가 감소할 것으로 전망했는데요. 가트너는 이같은 수요 부진으로 애플을 비롯한 스마트폰 제조사부터 엔비디아 TSMC 같은 반도체 업체까지 압력이 가해질 것이라고 진단했습니다. ◇ EU 가상자산 돈세탁 막는다 유럽연합이 가상자산을 이용한 돈세탁을 막기위해 관련 기업을 규제하는 방안에 잠정 합의했습니다. 잠정안에는 가상자산 업체가 당국에 모든 디지털자산 거래에 대한 신원 확인 정보를 제공하도록 하는 내용이 담겼는데요. 이에 따라 업체들은 관련 개인정보를 확보해야하고 당국이 이를 요구할 경우 제출해야 합니다. 또 거래액이 1천 유로 우리돈 130만 원을 넘길 경우 비인증 거래소가 관리하는 가상자산 지갑도 똑같은 규칙이 적용되는데요. 여기에 더해 송금 규제를 활용해 거래를 상시 추적하고 불법성이 의심되는 거래를 막을 수 있도록 할 방침입니다. 이와 관련해 미국 최대 가상자산 거래소 코인베이스 등 관련 기업 40여 곳은 개인정보 침해 가능성을 언급하며 줄곧 반대 입장을 밝혀왔는데요. 하지만 최근 가상자산 관련 범죄 사례가 급증하고 있는 만큼 규제 움직임이 힘을

In [5]:
import pandas as pd
df = economy_dataset.to_pandas()
df.head()

,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...


## sLLM 답변데이터 생성
llm을 이용해서 sLLM이 답변했으면 하는 내용을 생성해낸다. 이때 답변을 품질이 중요하므로, 되도록 상위모델을 사용하는 것이 좋다.

In [6]:
# 금융뉴스 분석용 구조화 출력 스키마(Pydantic) + 프롬프트 템플릿 구성
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate

class StockAnalysis(BaseModel):
    stock_related: bool = Field(description='뉴스와 주식 종목간의 연관성 여부')
    summary: str = Field(description='뉴스 요약')
    
    positive_stocks: List[str] = Field(description='긍정적 영향이 예상되는 주식 종목명 목록', default_factory=list)
    positive_keywords: List[str] = Field(description='긍정적 영향의 근거가 되는 키워드 목록', default_factory=list)
    positive_reasons: List[str] = Field(description='긍정적 영향이 예상되는 이유')
    
    negative_stocks: List[str] = Field(description='부정적 영향이 예상되는 주식 종목명 목록', default_factory=list)
    negative_keywords: List[str] = Field(description='부정적 영향의 근거가 되는 키워드 목록', default_factory=list)
    negative_reasons: List[str] = Field(description='부정적 영향이 예상되는 이유')
    
    system_prompt = '''  # 모델 역할/출력 규칙을 고정하는 시스템 프롬프트
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''

user_prompt = '''  # 분석 대상 뉴스 본문을 전달하는 사용자 프롬프트
다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.

[news]
{news}
'''

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human',user_prompt)
])
news = df['document'][1]
prompt.invoke({'news':news})

PydanticUserError: A non-annotated attribute was detected: `system_prompt = "  # 모델 역할/출력 규칙을 고정하는 시스템 프롬프트\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n"`. All model fields require a type annotation; if `system_prompt` is not meant to be a field, you may be able to resolve this error by annotating it as a `ClassVar` or updating `model_config['ignored_types']`.

For further information visit https://errors.pydantic.dev/2.12/u/model-field-missing-annotation